In [1]:
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
from natsort import natsorted
from scipy.stats import pearsonr
from os.path import join as pjoin
from dask.distributed import LocalCluster, Client

sys.path.append('../../')
import circletrack_neural as ctn
import circletrack_behavior as ctb
import place_cells as pc

import dask
from dask.diagnostics import ProgressBar

In [2]:
## Settings 
project_dir = 'MultiCon_Imaging'
experiment_dir = 'MultiCon_Imaging5'
mouse_list = ['mc44', 'mc46', 'mc49', 'mc51', 'mc52']
mouse_list = ['mc44']
dpath = f'../../../{project_dir}/{experiment_dir}/output/aligned_minian/'
spath = f'../../../{project_dir}/{experiment_dir}/output/aligned_place_cells_min_trials/'
crossreg_path = f'../../../{project_dir}/{experiment_dir}/output/cross_registration_results'

data_type = 'S'
only_running = True
correct_dir = True
velocity_thresh = 10
bin_size = 0.16
centroid_distance = 4 ## for cross-registration
reference_session = '1'
comparisons = ['1', '2', '3', '4', '5']

file_str = f'mappings_{centroid_distance}_None.pkl' ## cross-registration file with all days

xr.set_options(keep_attrs=True)

In [3]:
## Load cluster
cluster = LocalCluster(
    n_workers=4,
    memory_limit='2GB',
    resources={'MEM': 1},
    threads_per_worker=2,
    dashboard_address=':9001'
)
client = Client(cluster)

In [ ]:
## Compare reference data's tuning curves to rotated data's tuning curves
for mouse in mouse_list:
    mpath = pjoin(dpath, f'{mouse}/{data_type}')
    ref_ses_path = pjoin(mpath, f'{mouse}_{data_type}_{reference_session}.nc')
    comparison_paths = [pjoin(mpath, f'{mouse}_{data_type}_{s}.nc') for s in comparisons]
        
    mappings = pd.read_pickle(pjoin(crossreg_path, f'circletrack_data/{mouse}/{file_str}'))
    mappings.columns = mappings.columns.droplevel(0) ## remove Multi-Index
    
    ref = xr.open_dataset(ref_ses_path)[data_type] ## load reference data array
    ref_data, position_data = ctn.subset_correct_dir_and_running(ref, correct_dir=correct_dir, only_running=only_running, velocity_thresh=velocity_thresh)

    for session in comparison_paths:
        print(session)
        ldata = xr.open_dataset(session)[data_type]
        shared_cells = mappings[[ref.attrs['date'], ldata.attrs['date']]].dropna().reset_index(drop=True)
        loop_data, loop_position = ctn.subset_correct_dir_and_running(ldata, correct_dir=correct_dir, only_running=only_running, velocity_thresh=velocity_thresh)

In [ ]:
center = ctb.find_center(output['x'].values, output['y'].values)
for rotation in np.arange(0, (360 + degree_increment), degree_increment):
    rotated_points = np.empty(shape=(2, output['x'].shape[0]))
    for idx, (x, y) in enumerate(zip(output['x'].values, output['y'].values)):
        p = (x, y)
        rot = ctb.rotate(p, origin=center, degrees=rotation)
        rotated_points[0, idx] = rot[0] ## x values
        rotated_points[1, idx] = rot[1] ## y values

In [4]:
client.close()
cluster.close()